In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **01. Read data**

In [22]:
%%capture
!pip install rank-bm25 pyvi pyarrow

In [23]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/Legal_Sytem/01_Dataset/02_processed_data")

CORPUS_PATH = DATA_DIR / 'corpus' / 'corpus.parquet'
QUERIES_DIR = DATA_DIR / 'corpus' / 'queries.parquet'
QRELS_DIR = DATA_DIR / 'corpus' / 'qrels.parquet'

OUTPUT_DIR = Path("/content/drive/MyDrive/Legal_Sytem/02_Retriever/02_Baseline/BM25/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K_LIST = [1, 3, 5, 10, 15, 20, 25, 30]
MAX_TOP_K = max(TOP_K_LIST)

# Số ứng viên lưu lại cho bước hybrid (mục 5 của đề tài) và bước mine hard negative.
RANKING_DEPTH = 50

In [24]:
corpus = pd.read_parquet(CORPUS_PATH)
corpus.head(2)

,doc_id,title,document,article,context,qa_count,content_hash,embedding_text
0,DOC_40231d053f0f29e1bb80,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG - Điều ...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG,Điều 9. Tuyển bổ sung và loại ra khỏi chương t...,Điều 9.\tTuyển bổ sung và loại ra khỏi chương...,70,CONTENT_7fce8ab9b5ca93015231,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂN...
1,DOC_91760158c35d7a2cf252,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,Điều 4. Kiểm tra xếp lớp đầu khóa cho sinh viê...,Điều 4. Kiểm tra xếp lớp đầu khóa cho ...,68,CONTENT_8ce05a028a6c8e694978,Văn bản: QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ...


In [25]:
query = pd.read_parquet(QUERIES_DIR)
qrels = pd.read_parquet(QRELS_DIR)

In [26]:
print('Corpus:\n', corpus.shape)
print('Corpus columns:\n', corpus.columns.tolist())
print('Query:\n', query.shape)
print('Query columns:\n', query.columns.tolist())
print('Qrels:\n', qrels.shape)
print('Qrels columns:\n', qrels.columns.tolist())

Corpus:
 (294, 8)
Corpus columns:
 ['doc_id', 'title', 'document', 'article', 'context', 'qa_count', 'content_hash', 'embedding_text']
Query:
 (9588, 3)
Query columns:
 ['query_id', 'query', 'split']
Qrels:
 (9609, 4)
Qrels columns:
 ['query_id', 'doc_id', 'split', 'relevance']


In [27]:
assert corpus['doc_id'].is_unique, "corpus có doc_id trùng."
assert corpus['content_hash'].is_unique, (
    "Còn passage trùng nội dung nhưng khác doc_id — chạy lại Preprocessing.ipynb bản mới."
)
assert query['query_id'].is_unique, "queries có query_id trùng."
assert qrels['doc_id'].isin(set(corpus['doc_id'])).all(), "qrels trỏ tới doc_id không có trong corpus."
assert qrels['query_id'].isin(set(query['query_id'])).all(), "qrels trỏ tới query_id không có trong queries."

print(f"Corpus: {len(corpus):,} passage")

for split_name in ['train', 'val', 'test']:
  print(f"{split_name}: {int((query['split'] == split_name).sum()):,} câu hỏi")

Corpus: 294 passage
train: 7,697 câu hỏi
val: 933 câu hỏi
test: 958 câu hỏi


## **02. Tokenizer**

In [28]:
from pyvi import ViTokenizer
from rank_bm25 import BM25Okapi
import re


def tokenizer_vietnamese(text: str) -> list[str]:
  text = str(text).strip().lower()

  if not text:
    return []

  text = re.sub(r"[^\w\s]", " ", text)

  segmented = ViTokenizer.tokenize(text)

  return segmented.split()

In [29]:
corpus_texts = corpus['context'].tolist()

tokenized_corpus = [ tokenizer_vietnamese(text) for text in corpus_texts]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built.")
print("Number of documents: ", len(tokenized_corpus))

BM25 index built.
Number of documents:  294


In [30]:
print(tokenizer_vietnamese("Sinh viên dự bị không trở thành sinh viên chính thức bao nhiêu học kỳ?"))

['sinh_viên', 'dự_bị', 'không', 'trở_thành', 'sinh_viên', 'chính_thức', 'bao_nhiêu', 'học_kỳ']


## **03. Retriever**

### **3.1. Build BM25 Index**

In [31]:
def build_bm25_index(corpus: pd.DataFrame, text_column: str = 'embedding_text', k1: float = 1.5, b: float = 0.75):
  tokenized_corpus = [tokenizer_vietnamese(text) for text in corpus[text_column]]

  bm25_index = BM25Okapi(tokenized_corpus, k1=k1, b=b)

  doc_id_array = corpus['doc_id'].to_numpy()

  return bm25_index, doc_id_array

In [32]:
bm25_index, doc_id_array = build_bm25_index(corpus, text_column='embedding_text')

print("BM25 index built.")
print("Number of documents:", len(doc_id_array))

BM25 index built.
Number of documents: 294


### **3.2. Truy xuất và lưu ranking**

In [33]:
import numpy as np

def retrieve(query_text: str, bm25_index, doc_id_array: np.ndarray, top_k: int = RANKING_DEPTH):
  tokenized_query = tokenizer_vietnamese(query_text)

  scores = np.asarray(bm25_index.get_scores(tokenized_query))

  # kind='stable' để các doc bằng điểm luôn giữ thứ tự corpus -> kết quả tái lập được.
  top_indices = np.argsort(-scores, kind='stable')[:top_k]

  return doc_id_array[top_indices], scores[top_indices]

In [34]:
def retrieve_all_queries(queries_dataframe: pd.DataFrame, bm25_index, doc_id_array: np.ndarray, top_k: int = RANKING_DEPTH) -> pd.DataFrame:
  ranking_records = []

  for query_id, query_text, split_name in zip(
      queries_dataframe['query_id'], queries_dataframe['query'], queries_dataframe['split']
  ):
    retrieved_doc_ids, retrieved_scores = retrieve(query_text, bm25_index, doc_id_array, top_k)

    for rank, (doc_id, score) in enumerate(zip(retrieved_doc_ids, retrieved_scores), start=1):
      ranking_records.append({
          'query_id': query_id,
          'doc_id': doc_id,
          'rank': rank,
          'score': float(score),
          'split': split_name,
      })

  return pd.DataFrame(ranking_records)

### **3.3. Test sample**

In [35]:
sample_query_row = query[query['split'] == 'val'].iloc[0]

retrieved_doc_ids, retrieved_scores = retrieve(sample_query_row['query'], bm25_index, doc_id_array, top_k=10)

gold_relevant_doc_ids = set(qrels.loc[qrels['query_id'] == sample_query_row['query_id'], 'doc_id'])

print("Query:", sample_query_row['query'])
print("Gold :", gold_relevant_doc_ids)
print()

sample_result = pd.DataFrame({
    'rank': np.arange(1, len(retrieved_doc_ids) + 1),
    'doc_id': retrieved_doc_ids,
    'score': retrieved_scores.round(4),
    'is_gold': [doc_id in gold_relevant_doc_ids for doc_id in retrieved_doc_ids],
})

sample_result = sample_result.merge(corpus[['doc_id', 'title']], on='doc_id', how='left')

display(sample_result)

Query: Sinh viên chưa hết thời gian tối đa hoàn thành khóa học quy định tại Điều 6 của Quy chế này, đã hoàn thành các học phần trong chương trình đào tạo có nguyện vọng xin thôi học theo diện này thì phải làm gì?
Gold : {'DOC_e08b1c04f5664b3bd801'}



,rank,doc_id,score,is_gold,title
0,1,DOC_e08b1c04f5664b3bd801,60.5377,True,QUY CHẾ ĐÀO TẠO THEO HỌC CHẾ TÍN CHỈ CHO HỆ ĐẠ...
1,2,DOC_9636ab608cab98c30952,44.3453,False,QUY CHẾ ĐÀO TẠO THEO HỌC CHẾ TÍN CHỈ CHO HỆ ĐẠ...
2,3,DOC_82965f5d3b8ba282e106,41.5972,False,QUY ĐỊNH TẠM THỜI VỀ ĐÀO TẠO LIÊN THÔNG TỪ TRÌ...
3,4,DOC_62d595b87d2d61438805,40.4554,False,QUY CHẾ ĐÀO TẠO THEO HỌC CHẾ TÍN CHỈ CHO HỆ ĐẠ...
4,5,DOC_34542e6b5ad5aca6b25e,40.1249,False,QUY CHẾ ĐÀO TẠO THEO HỌC CHẾ TÍN CHỈ CHO HỆ ĐẠ...
5,6,DOC_0a4ed43bcf45af32dc0f,33.6676,False,QUY CHẾ ĐÀO TẠO THEO HỌC CHẾ TÍN CHỈ CHO HỆ ĐẠ...
6,7,DOC_12e6eafacf18139b9eb4,33.3139,False,QUY CHẾ ĐÀO TẠO THEO HỌC CHẾ TÍN CHỈ CHO HỆ ĐẠ...
7,8,DOC_c331195ebd0473b185c9,32.5232,False,QUY CHẾ ĐÀO TẠO TỪ XA TRÌNH ĐỘ ĐẠI HỌC CỦA TRƯ...
8,9,DOC_2bad0739b99f0b1d711e,31.1972,False,QUY ĐỊNH về việc tổ chức dạy – học ngoài giờ h...
9,10,DOC_0e82710ba3f01da9b6aa,30.7719,False,QUY CHẾ ĐÀO TẠO THEO HỌC CHẾ TÍN CHỈ CHO HỆ ĐẠ...


## **4. Danh gia Evaluation**

### **4.1. Hàm tính chỉ số**

In [36]:
from collections import defaultdict

def build_qrels_map(qrels_dataframe: pd.DataFrame) -> dict:
  qrels_map = defaultdict(set)

  for query_id, doc_id, relevance in zip(qrels_dataframe['query_id'], qrels_dataframe['doc_id'], qrels_dataframe['relevance']):
    if relevance > 0:
      qrels_map[query_id].add(doc_id)

  return dict(qrels_map)

Thêm `hit_rate_at_k` (còn gọi là Accuracy@k) bên cạnh `recall_at_k`: với câu hỏi 1 positive thì hai
chỉ số bằng nhau, nhưng dữ liệu này có 20 câu hỏi nhiều positive nên chúng sẽ lệch nhau ở đó.

In [37]:
def precision_at_k(gold_relevant_doc_ids: set, retrieved_doc_ids: list, k: int) -> float:
  if not gold_relevant_doc_ids or k <= 0:
        return 0.0
  top_k_retrieved = set(retrieved_doc_ids[:k])
  relevant_retrieved = top_k_retrieved & gold_relevant_doc_ids

  return len(relevant_retrieved) / k

In [38]:
def recall_at_k(gold_relevant_doc_ids: set, retrieved_doc_ids: list, k: int) -> float:
  if not gold_relevant_doc_ids:
    return 0.0

  return len(set(retrieved_doc_ids[:k]) & gold_relevant_doc_ids) / len(gold_relevant_doc_ids)

# Trong top_K co tai lieu dung hay khong
def hit_rate_at_k(gold_relevant_doc_ids: set, retrieved_doc_ids: list, k: int) -> float:
  return float(bool(set(retrieved_doc_ids[:k]) & gold_relevant_doc_ids))

In [39]:
def reciprocal_rank_at_k(gold_relevant_doc_ids: set, retrieved_doc_ids: list, k: int) -> float:
  # [:k] là dòng bị thiếu ở bản trước.
  for rank, doc_id in enumerate(retrieved_doc_ids[:k], start=1):
    if doc_id in gold_relevant_doc_ids:
      return 1.0 / rank
  return 0.0



def ndcg_at_k(gold_relevant_doc_ids: set, retrieved_doc_ids: list, k: int) -> float:
  if not gold_relevant_doc_ids:
    return 0.0

  # DCG@k
  actual_dcg = sum(1.0 / np.log2(rank + 1)
      for rank, doc_id in enumerate(retrieved_doc_ids[:k], start=1)
      if doc_id in gold_relevant_doc_ids)

  # nDCG@k
  ideal_dcg = sum(1.0 / np.log2(rank + 1)
      for rank in range(1, min(len(gold_relevant_doc_ids), k) + 1))

  return actual_dcg / ideal_dcg if ideal_dcg else 0.0

### **4.2. Đánh giá từ file ranking**

In [40]:
def evaluation(ranking_dataframe: pd.DataFrame, qrels_map: dict, top_k_list: list[int] = TOP_K_LIST) -> pd.DataFrame:
  ranking_dataframe = ranking_dataframe.sort_values(['query_id', 'rank'])

  # {query: [doc_1, doc_2,...]}
  ranked_doc_ids_by_query = (
      ranking_dataframe.groupby('query_id')['doc_id'].apply(list).to_dict()
  )

  records = []

  for query_id, retrieved_doc_ids in ranked_doc_ids_by_query.items():
    gold_relevant_doc_ids = qrels_map.get(query_id)

    if not gold_relevant_doc_ids:
      continue

    record = {'query_id': query_id, 'number_of_positives': len(gold_relevant_doc_ids)}

    for k in top_k_list:
      record[f'recall@{k}'] = recall_at_k(gold_relevant_doc_ids, retrieved_doc_ids, k)
      record[f'hit@{k}'] = hit_rate_at_k(gold_relevant_doc_ids, retrieved_doc_ids, k)
      record[f'mrr@{k}'] = reciprocal_rank_at_k(gold_relevant_doc_ids, retrieved_doc_ids, k)
      record[f'ndcg@{k}'] = ndcg_at_k(gold_relevant_doc_ids, retrieved_doc_ids, k)

    records.append(record)

  return pd.DataFrame(records)


def summarize(per_query_dataframe: pd.DataFrame) -> pd.Series:
  return per_query_dataframe.drop(columns=['query_id', 'number_of_positives']).mean()

## **5. Finetune trên tập val**

### **5.1. Chọn trường để index**

In [41]:
field_comparison_records = []

validation_query = query[query['split'] == 'val']
validation_qrels_map = build_qrels_map(qrels[qrels['split'] == 'val'])

for text_column in ['context', 'embedding_text']:
  candidate_index, candidate_doc_id_array = build_bm25_index(corpus, text_column=text_column)

  candidate_ranking = retrieve_all_queries(validation_query, candidate_index, candidate_doc_id_array, top_k=MAX_TOP_K)

  candidate_summary = summarize(evaluation(candidate_ranking, validation_qrels_map))

  field_comparison_records.append({
      'text_column': text_column,
      'recall@1': candidate_summary['recall@1'],
      'recall@3': candidate_summary['recall@3'],
      'recall@5': candidate_summary['recall@5'],
      'recall@10': candidate_summary['recall@10'],
      'mrr@10': candidate_summary['mrr@10'],
      'ndcg@10': candidate_summary['ndcg@10'],
  })

field_comparison_df = pd.DataFrame(field_comparison_records).round(4)

display(field_comparison_df)

BEST_TEXT_COLUMN = field_comparison_df.loc[field_comparison_df['mrr@10'].idxmax(), 'text_column']

print("Trường được chọn:", BEST_TEXT_COLUMN)

,text_column,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10
0,context,0.6956,0.8628,0.9025,0.9378,0.7844,0.8222
1,embedding_text,0.7095,0.8714,0.9110,0.9486,0.7953,0.8329


Trường được chọn: embedding_text


### **5.3. Grid Search k1 và b**

In [50]:
K1_GRID = [0.9, 1.2, 1.5, 1.8, 2.1]
B_GRID = [0.3, 0.5, 0.65, 0.75, 0.9]

grid_search_records = []

for k1 in K1_GRID:
  for b in B_GRID:
    candidate_index, candidate_doc_id_array = build_bm25_index(
        corpus, text_column=BEST_TEXT_COLUMN, k1=k1, b=b
    )

    candidate_ranking = retrieve_all_queries(
        validation_query, candidate_index, candidate_doc_id_array, top_k=MAX_TOP_K
    )

    candidate_summary = summarize(evaluation(candidate_ranking, validation_qrels_map))

    grid_search_records.append({
        'k1': k1,
        'b': b,
        'recall@1': candidate_summary['recall@1'],
        'recall@3': candidate_summary['recall@3'],
        'recall@5': candidate_summary['recall@5'],
        'recall@10': candidate_summary['recall@10'],
        'mrr@10': candidate_summary['mrr@10'],
        'ndcg@10': candidate_summary['ndcg@10'],
    })

grid_search_df = pd.DataFrame(grid_search_records).round(4)

display(grid_search_df.sort_values('mrr@10', ascending=False).head(10))

grid_search_df.to_csv(OUTPUT_DIR / 'bm25_grid_search.csv', index=False, encoding='utf-8-sig')

,k1,b,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10
4,0.9,0.90,0.7128,0.8703,0.9089,0.9464,0.7970,0.8337
8,1.2,0.75,0.7095,0.8650,0.9132,0.9486,0.7965,0.8339
13,1.5,0.75,0.7095,0.8714,0.9110,0.9486,0.7953,0.8329
9,1.2,0.90,0.7085,0.8671,0.9121,0.9496,0.7944,0.8324
7,1.2,0.65,0.7053,0.8660,0.9164,0.9464,0.7943,0.8318
3,0.9,0.75,0.7042,0.8639,0.9143,0.9453,0.7934,0.8308
14,1.5,0.90,0.7042,0.8660,0.9100,0.9486,0.7919,0.8303
2,0.9,0.65,0.7020,0.8639,0.9089,0.9464,0.7907,0.8290
12,1.5,0.65,0.6988,0.8660,0.9132,0.9443,0.7900,0.8281
18,1.8,0.75,0.6945,0.8703,0.9143,0.9475,0.7873,0.8268


In [51]:
best_row = grid_search_df.loc[grid_search_df['mrr@10'].idxmax()]

BEST_K1 = float(best_row['k1'])
BEST_B = float(best_row['b'])

print(f"Cấu hình tốt nhất trên val: text_column = {BEST_TEXT_COLUMN}, \nk1 = {BEST_K1}, b = {BEST_B}")
print(f"\nrecall@1 = {best_row['recall@1']:.4f}  mrr@10 = {best_row['mrr@10']:.4f}  ndcg@10 = {best_row['ndcg@10']:.4f}")

# So sánh với cấu hình mặc định để biết việc tinh chỉnh có đáng hay không.
default_row = grid_search_df[(grid_search_df['k1'] == 1.5) & (grid_search_df['b'] == 0.75)].iloc[0]

print(f"Mặc định (k1=1.5, b=0.75): recall@1 = {default_row['recall@1']:.4f}  mrr@10 = {default_row['mrr@10']:.4f}")

Cấu hình tốt nhất trên val: text_column = embedding_text, 
k1 = 0.9, b = 0.9

recall@1 = 0.7128  mrr@10 = 0.7970  ndcg@10 = 0.8337
Mặc định (k1=1.5, b=0.75): recall@1 = 0.7095  mrr@10 = 0.7953


## **6. Results**

In [52]:
bm25_index, doc_id_array = build_bm25_index(
    corpus, text_column=BEST_TEXT_COLUMN, k1=BEST_K1, b=BEST_B
)

print(f"Đã dựng lại index với text_column={BEST_TEXT_COLUMN}, k1={BEST_K1}, b={BEST_B}")

Đã dựng lại index với text_column=embedding_text, k1=0.9, b=0.9


In [53]:
ranking_by_split = {}
per_query_by_split = {}
summary_records = []

for split_name in ['val', 'test']:
  split_query = query[query['split'] == split_name]
  split_qrels_map = build_qrels_map(qrels[qrels['split'] == split_name])

  split_ranking = retrieve_all_queries(split_query, bm25_index, doc_id_array, top_k=RANKING_DEPTH)

  split_per_query = evaluation(split_ranking, split_qrels_map)

  ranking_by_split[split_name] = split_ranking
  per_query_by_split[split_name] = split_per_query

  summary_records.append({
      'split': split_name,
      'number_of_queries': len(split_per_query),
      **summarize(split_per_query).to_dict(),
  })

summary_df = pd.DataFrame(summary_records).set_index('split')

display(summary_df[[column for column in summary_df.columns if column.startswith(('recall', 'mrr', 'ndcg'))]].round(4).T)

split,val,test
recall@1,0.7128,0.6691
mrr@1,0.7128,0.6691
ndcg@1,0.7128,0.6691
recall@3,0.8703,0.8497
mrr@3,0.7830,0.7523
ndcg@3,0.8054,0.7774
recall@5,0.9089,0.8904
mrr@5,0.7916,0.7618
ndcg@5,0.8212,0.7944
recall@10,0.9464,0.9290


In [54]:
# Kiểm tra tính đúng đắn: với câu hỏi 1 positive thì MRR@1 phải bằng Recall@1.
for split_name, per_query_dataframe in per_query_by_split.items():
  single_positive = per_query_dataframe[per_query_dataframe['number_of_positives'] == 1]

  assert np.allclose(single_positive['mrr@1'], single_positive['recall@1']), (f"{split_name}: MRR@1 khác Recall@1 — hàm reciprocal_rank_at_k có lỗi.")

  # MRR phải tăng đơn điệu theo k. Nếu bằng nhau hết là dấu hiệu quên cắt [:k].
  mean_mrr = [per_query_dataframe[f'mrr@{k}'].mean() for k in TOP_K_LIST]

  assert mean_mrr == sorted(mean_mrr), f"{split_name}: MRR không tăng theo k."
  assert len(set(np.round(mean_mrr, 6))) > 1, f"{split_name}: MRR không đổi theo k — vẫn còn lỗi cũ."

print("Các phép kiểm tra chỉ số đã pass.")

Các phép kiểm tra chỉ số đã pass.


### **6.1. Latency**

Đánh giá model qua hiệu suất và tốc độ, nên cần đo cho BM25

In [55]:
import time

latency_query_list = validation_query['query'].tolist()[:200]

# Chạy nóng trước để loại chi phí khởi tạo lần đầu của ViTokenizer.
for query_text in latency_query_list[:10]:
  retrieve(query_text, bm25_index, doc_id_array, top_k=MAX_TOP_K)

start_time = time.perf_counter()

for query_text in latency_query_list:
  retrieve(query_text, bm25_index, doc_id_array, top_k=MAX_TOP_K)

elapsed_time = time.perf_counter() - start_time

latency_report = {
    'retriever': 'BM25Okapi',
    'text_column': BEST_TEXT_COLUMN,
    'k1': BEST_K1,
    'b': BEST_B,
    'corpus_size': len(corpus),
    'number_of_queries': len(latency_query_list),
    'latency_ms_per_query': round(elapsed_time / len(latency_query_list) * 1000, 3),
    'queries_per_second': round(len(latency_query_list) / elapsed_time, 1),
}

print(json.dumps(latency_report, ensure_ascii=False, indent=2))

{
  "retriever": "BM25Okapi",
  "text_column": "embedding_text",
  "k1": 0.9,
  "b": 0.9,
  "corpus_size": 294,
  "number_of_queries": 200,
  "latency_ms_per_query": 1.332,
  "queries_per_second": 750.9
}


### **6.2. Save**

In [56]:
summary_df.round(6).to_csv(OUTPUT_DIR / 'bm25_summary.csv', encoding='utf-8-sig')

for split_name, per_query_dataframe in per_query_by_split.items():
  per_query_dataframe.to_parquet(OUTPUT_DIR / f'{split_name}_metrics_per_query.parquet', index=False)

for split_name, split_ranking in ranking_by_split.items():
  split_ranking.to_parquet(OUTPUT_DIR / f'{split_name}_ranking.parquet', index=False)

with open(OUTPUT_DIR / 'bm25_config.json', 'w', encoding='utf-8') as file:
  json.dump(
      {
          'text_column': BEST_TEXT_COLUMN,
          'k1': BEST_K1,
          'b': BEST_B,
          'ranking_depth': RANKING_DEPTH,
          'top_k_list': TOP_K_LIST,
          **latency_report,
      },
      file, ensure_ascii=False, indent=2
  )

print("Đã lưu summary, per-query metrics, ranking và config.")

Đã lưu summary, per-query metrics, ranking và config.


## **07. Ranking trên train**

In [57]:
train_query = query[query['split'] == 'train']

train_ranking = retrieve_all_queries(train_query, bm25_index, doc_id_array, top_k=RANKING_DEPTH)

# Đánh dấu positive để bước 4 lọc ra ngay, không phải join lại.
train_positive_pairs = set(
    zip(qrels.loc[qrels['split'] == 'train', 'query_id'], qrels.loc[qrels['split'] == 'train', 'doc_id'])
)

train_ranking['is_positive'] = [
    (query_id, doc_id) in train_positive_pairs
    for query_id, doc_id in zip(train_ranking['query_id'], train_ranking['doc_id'])
]

train_ranking.to_parquet(OUTPUT_DIR / 'train_ranking.parquet', index=False)

print(f"train_ranking: {len(train_ranking):,} dòng cho {train_query['query_id'].nunique():,} câu hỏi")
print(f"Số cặp positive nằm trong top-{RANKING_DEPTH}: {int(train_ranking['is_positive'].sum()):,}")

train_ranking: 384,850 dòng cho 7,697 câu hỏi
Số cặp positive nằm trong top-50: 7,586
